# 06 — Latent Diffusion

Latent diffusion runs the diffusion process in a compressed latent space, enabling high-resolution generation at tractable compute.

## Moving to Latent Space

Pixel-space diffusion (DDPM on 256×256 images) requires running a UNet on millions of pixels per step for hundreds of steps. Latent diffusion (Rombach et al., 2022) separates:
1. **Perceptual compression** (VAE): encoder *E* maps images to a compact latent *z = E(x)*; decoder *D* reconstructs *x = D(z)*. The VAE is trained once and frozen.
2. **Diffusion in latent space**: the noise prediction network operates on *z* (typically 4-8x smaller than the image in each spatial dimension), dramatically reducing compute.

The final decoder adds back fine-grained perceptual detail that the diffusion model doesn't need to model.

The latent space has a key property: it is **semantically smooth** — nearby points decode to similar images, and directions correspond to semantic attributes. This is why latent interpolation, inpainting, and text-conditioned editing all work well in the latent space.

Stable Diffusion uses an 8x downsampling VAE: a 512×512 image becomes a 64×64×4 latent, reducing diffusion compute by 64×.

In [ ]:
# Latent diffusion: small-scale demonstration
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)

# Step 1: Train a small VAE as the perceptual compressor
# Input: 8-dim features; latent: 2-dim
class VAE(nn.Module):
    def __init__(self, input_dim=8, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU())
        self.mu_head = nn.Linear(16, latent_dim)
        self.logvar_head = nn.Linear(16, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16), nn.ReLU(),
            nn.Linear(16, 32), nn.ReLU(),
            nn.Linear(32, input_dim)
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.mu_head(h), self.logvar_head(h)

    def reparameterise(self, mu, logvar):
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterise(mu, logvar)
        return self.decoder(z), mu, logvar

# Dataset: 3 clusters in 8D
from sklearn.datasets import make_blobs
X_np, y_np = make_blobs(n_samples=2000, n_features=8, centers=3, random_state=42)
X_np = (X_np - X_np.mean(0)) / X_np.std(0)
X_data = torch.tensor(X_np, dtype=torch.float32)

vae = VAE()
opt_vae = torch.optim.Adam(vae.parameters(), lr=3e-3)
for _ in range(500):
    recon, mu, logvar = vae(X_data)
    r_loss = ((recon - X_data)**2).mean()
    kl = -0.5 * (1 + logvar - mu**2 - logvar.exp()).mean()
    loss = r_loss + 0.1 * kl
    opt_vae.zero_grad(); loss.backward(); opt_vae.step()

vae.eval()
with torch.no_grad():
    mus, _ = vae.encode(X_data)
    z_data = mus
print(f'VAE trained. Reconstruction error: {r_loss.item():.4f}')
print(f'Latent space shape: {z_data.shape}')

In [ ]:
# Step 2: Train diffusion model in the latent space
T = 200
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

class LatentNoiseNet(nn.Module):
    def __init__(self, latent_dim=2, T=200):
        super().__init__()
        self.time_embed = nn.Embedding(T, 32)
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 32, 64), nn.GELU(),
            nn.Linear(64, 32), nn.GELU(),
            nn.Linear(32, latent_dim)
        )
    def forward(self, z, t):
        return self.net(torch.cat([z, self.time_embed(t)], dim=-1))

latent_net = LatentNoiseNet()
opt_ldm = torch.optim.Adam(latent_net.parameters(), lr=3e-4)

for step in range(2000):
    idx = torch.randint(0, len(z_data), (256,))
    z0 = z_data[idx].detach()
    t = torch.randint(0, T, (256,))
    noise = torch.randn_like(z0)
    z_t = alpha_bars[t].sqrt().unsqueeze(1) * z0 + (1-alpha_bars[t]).sqrt().unsqueeze(1) * noise
    loss = ((latent_net(z_t, t) - noise)**2).mean()
    opt_ldm.zero_grad(); loss.backward(); opt_ldm.step()

# Step 3: Sample new data
@torch.no_grad()
def ldm_sample(latent_net, vae_decoder, n_samples):
    z = torch.randn(n_samples, 2)
    for t_val in range(T-1, -1, -1):
        t_b = torch.full((n_samples,), t_val, dtype=torch.long)
        eps = latent_net(z, t_b)
        mean = (1/alphas[t_val].sqrt()) * (z - betas[t_val]/(1-alpha_bars[t_val]).sqrt() * eps)
        if t_val > 0:
            z = mean + betas[t_val].sqrt() * torch.randn_like(z)
        else:
            z = mean
    return vae_decoder(z)

samples = ldm_sample(latent_net, vae.decoder, 300)
print(f'Generated {samples.shape[0]} samples in 8D space')

# Visualise first 2 dims
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.scatter(X_np[:300, 0], X_np[:300, 1], c=y_np[:300], alpha=0.3, s=5, cmap='tab10')
ax1.set_title('Real data (dim 0, 1)')
ax2.scatter(samples[:, 0].numpy(), samples[:, 1].numpy(), alpha=0.3, s=5, color='purple')
ax2.set_title('LDM samples (dim 0, 1)')
plt.suptitle('Latent Diffusion Model')
plt.tight_layout()
plt.savefig('/tmp/ldm_samples.png', dpi=100, bbox_inches='tight')
plt.show()
print('LDM demo complete')